In [ ]:
import os
import pandas as pd
import glob

from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *


def merge_gibuu_subfiles(directory, keys2load):
    """
    Finds all cc1pi_GIBUU_subfileX.df files, prints their info,
    and merges them into a single dictionary of DataFrames.
    """
    
    # Find all .df files in the scratch directory
    file_pattern = os.path.join(directory, "cc1pi_GIBUU_subfile*.df")
    files = sorted(glob.glob(file_pattern))
    
    if not files:
        print("No subfiles found in the directory!")
        return None

    merged_data = {k: [] for k in keys2load}
    total_events = 0
    total_pot = 0.0

    print(f"Found {len(files)} subfiles. Starting merge...")

    for f in files:
        try:
            # Load the subfile
            df_dict = load_df(f, keys2load, 100)
            
            # Print info for this chunk
            n_evts = len(df_dict['cc1pi'])
            pot = df_dict['hdr']['pot'].sum()
            print(f"File: {os.path.basename(f)} | Events: {n_evts} | POT: {pot:.2e}")
            
            # Accumulate
            total_events += n_evts
            total_pot += pot
            for k in keys2load:
                merged_data[k].append(df_dict[k])
                
        except Exception as e:
            print(f"Error loading {f}: {e}")

    # Concatenate everything
    final_dict = {k: pd.concat(merged_data[k], ignore_index=False) for k in keys2load if merged_data[k]}
    
    print("\n" + "="*30)
    print("FINAL MERGE SUMMARY")
    print(f"Total Files Merged: {len(files)}")
    print(f"Total CC1pi Events: {total_events}")
    print(f"Total Combined POT: {total_pot:.2e}")
    print("="*30)
    
    return final_dict

# --- Usage ---
scratch_path = "/scratch/7DayLifetime/lpelegrina/GiBUUFiles"
target_keys = ["cc1pi", "hdr", "histpotdf", "nudf"]

# 1. Run the merge
mc_GIBUU = merge_gibuu_subfiles(scratch_path, target_keys)
print(f"ALL: | Events: { len(mc_GIBUU['cc1pi'])} | POT: {mc_GIBUU['hdr']['pot'].sum():.2e}")

output_path = "/scratch/7DayLifetime/lpelegrina/GiBUUFiles/mc_GIBUU.df"

print(f"Saving merged data to {output_path}...")

# Use 'w' to create/overwrite the file
with pd.HDFStore(output_path, mode='w') as store:
    for key in target_keys:
        if key in mc_GIBUU:
            df = mc_GIBUU[key]
            # 'fixed' format is what run_df_maker uses; it's fast and 
            # handles the multi-indices of CAFs well.
            store.put(key, df, format='fixed')
            print(f"  - Saved key '{key}' ({len(df)} rows)")

    # Crucial: Add the 'split' info so your load_df function 
    # knows this is a "single-split" file.
    split_info = pd.DataFrame({"n_split": [1]})
    store.put("split", split_info, format="fixed")

print("\nDone! The file is ready for use.")
print(f"\nSuccessfully saved merged dataframe to: {output_path}")
